In [3]:
import nanopub
print(nanopub.__version__)

2.1.0


In [1]:
import pandas as pd
flora = pd.read_csv("https://raw.githubusercontent.com/forrtproject/fred-data/refs/heads/main/output/flora.csv")


# FLoRA filtering

In [ ]:
flora_filtered = flora[flora["outcome"].isin(["successful", "failed", "mixed"])]


outcome
successful    1019
failed         989
mixed          625
Name: count, dtype: int64

In [64]:
# multi-study papers: several flora rows share one paper identifier but each has its own url_r.
# For these, work = url_r and the paper (doi_r or alt_identifier_r) goes into `collection`.

def shared_id_unique_url(df, key):
    """Rows whose `key` is shared across several rows, where every row in that
    group has a url_r and no two rows in the group have the same url_r."""
    d = df[df[key].notna()]
    g = d.groupby(key)["url_r"]
    qualifies = (
        (g.transform("size") > 1)                          # key shared across rows
        & (g.transform(lambda s: s.notna().all()))         # every row has a url_r
        & (g.transform("nunique") == g.transform("size"))  # url_r all different
    )
    return d[qualifies]

# A: doi_r shared across rows, each with a unique url_r
group_a = shared_id_unique_url(flora, "doi_r")
# B: no doi_r, alt_identifier_r shared across rows, each with a unique url_r
group_b = shared_id_unique_url(flora[flora["doi_r"].isna()], "alt_identifier_r")

multi_studies = pd.concat([group_a.assign(match="doi_r"),
                           group_b.assign(match="alt_identifier_r")])
multi_study_ids = set(multi_studies["id"])

print(len(group_a), "rows via doi_r,", len(group_b), "rows via alt_identifier_r")
multi_studies.sort_values(["doi_r", "alt_identifier_r"])[["id", "match", "doi_r", "alt_identifier_r", "url_r"]]
multi_studies[multi_studies["url_r"] == "https://osf.io/bfs6p"][["id", "match", "doi_r", "alt_identifier_r", "url_r"]   ]

61 rows via doi_r, 179 rows via alt_identifier_r


,id,match,doi_r,alt_identifier_r,url_r
2337,FLORA-002635,alt_identifier_r,NaN,10.1038/s41586-025-10078-y,https://osf.io/bfs6p


# test df for testing

In [101]:
test_ids = [
    "FLORA-000006", "FLORA-000007", "FLORA-000008", "FLORA-000009",
    "FLORA-000013", "FLORA-000014", "FLORA-000015", "FLORA-000017",
    "FLORA-000018", "FLORA-000019", "FLORA-000020", "FLORA-000022",
    "FLORA-000023", "FLORA-000027", "FLORA-000030", "FLORA-000036",
]

test = flora_filtered[flora_filtered["id"].isin(test_ids)]

print(len(test), "of", len(test_ids), "found")
test[["id", "doi_r", "alt_identifier_r", "url_r", "outcome"]]


16 of 16 found


,id,doi_r,alt_identifier_r,url_r,outcome
5,FLORA-000006,10.1080/02671522.2019.1678062,NaN,NaN,successful
6,FLORA-000007,10.1371/journal.pone.0294325,NaN,NaN,successful
7,FLORA-000008,10.31820/pt.28.3.7,NaN,NaN,successful
8,FLORA-000009,10.1080/10409289.2018.1475202,NaN,NaN,successful
12,FLORA-000013,10.1007/s12186-012-9090-0,NaN,NaN,successful
13,FLORA-000014,10.1002/bin.70067,NaN,NaN,successful
14,FLORA-000015,10.1007/s11469-021-00695-9,NaN,NaN,successful
16,FLORA-000017,10.3102/0013189x20940304,NaN,NaN,successful
17,FLORA-000018,10.1080/08993408.2022.2071543,NaN,NaN,mixed
18,FLORA-000019,10.1038/s41598-025-10331-4,NaN,NaN,successful


# Nanopubs

In [107]:
from rdflib import Graph, Literal, Namespace, URIRef, RDF, RDFS
from nanopub import Nanopub, NanopubConf, load_profile

TEMPLATE = URIRef("https://w3id.org/np/RA0-q9IdP_d6PPmIUeKjYLbm1wxsn_ga0gWTN1m-azxsI")
CITO  = Namespace("http://purl.org/spar/cito/")
FORRT = Namespace("https://w3id.org/sciencelive/o/terms/")
NT    = Namespace("https://w3id.org/np/o/ntemplate/")
DCT   = Namespace("http://purl.org/dc/terms/")
PROV = Namespace("http://www.w3.org/ns/prov#")
FLORA = URIRef("https://forrt.org/flora-replication-atlas/")
PROV_TEMPLATE = URIRef("https://w3id.org/np/RAGxHxC8z_PnSSrfMIBb5vWdhdlZAj_VZYyMt0C7ABOaM")


OUTCOME_TO_CITO = {
    "successful": CITO.confirms,
    "failed":     CITO.refutes,
    "mixed":      CITO.discusses,
}

def doi_uri(doi):
    """Turn flora identifiers into URIs: bare DOIs -> https://doi.org/..., other bare links -> https://..."""
    doi = str(doi).strip()
    if doi.startswith("http"):
        return URIRef(doi)
    if doi.startswith("10."):
        return URIRef("https://doi.org/" + doi)
    return URIRef("https://" + doi)

def require(row, *cols):
    for col in cols:
        if pd.isna(row[col]):
            raise ValueError(f"Required column {col!r} is empty")

def build_nanopub(row, conf, multi_ids=frozenset()):
    outcome = str(row["outcome"]).strip().lower()
    if outcome not in OUTCOME_TO_CITO:
        raise ValueError(f"Can't map outcome {row['outcome']!r} to a CiTO relation")
    require(row, "doi_o", "outcome_quote")

    if row["id"] in multi_ids:
        # multi-study paper: the individual study is the work, the paper is the collection
        require(row, "url_r")
        work = doi_uri(row["url_r"])
        paper = row["doi_r"] if pd.notna(row["doi_r"]) else row["alt_identifier_r"]
        collection = doi_uri(paper)
    else:
        require(row, "doi_r")
        work = doi_uri(row["doi_r"])
        collection = None
    cited = doi_uri(row["doi_o"])

    assertion = Graph()
    assertion.bind("cito", CITO)
    assertion.bind("forrt", FORRT)
    assertion.bind("dct", DCT)
    assertion.add((work, RDF.type, FORRT["FORRT-Replication-Study"]))               # st01
    assertion.add((work, OUTCOME_TO_CITO[outcome], cited))                          # st02
    assertion.add((work, FORRT["FORRT-Replication-Outcome"],
                   Literal(str(row["outcome_quote"]).strip())))                     # st03
    if collection is not None:                                                      # st04 (optional)
        assertion.add((work, DCT.isPartOf, collection))

    np = Nanopub(conf=conf, assertion=assertion)
   # provenance: follow the template's default provenance (attributed to creator + FLoRA)
    np.provenance.bind("prov", PROV)
    np.provenance.add((np.metadata.assertion, PROV.wasAttributedTo, FLORA))

    this = np.metadata.namespace[""]
    np.pubinfo.bind("nt", NT)
    np.pubinfo.add((this, NT.wasCreatedFromTemplate, TEMPLATE))
    np.pubinfo.add((this, NT.wasCreatedFromProvenanceTemplate, PROV_TEMPLATE))
    np.pubinfo.add((this, RDFS.label, Literal(f"Replication outcome of: {work}")))
    return np

np_conf = NanopubConf(
    profile=load_profile(),
    use_test_server=False,
    add_prov_generated_time=False,
    attribute_assertion_to_profile=True,
    attribute_publication_to_profile=True,
)


# build nanopubs for many flora entries
- start with a small batch on the test server
- rows with an unmappable outcome or missing required fields are skipped and reported
- duplicate rows (same work, original and outcome) are dropped so the same claim isn't published twice
- rows in `multi_study_ids` get work = `url_r` and collection = `doi_r` / `alt_identifier_r`

In [108]:
batch = test   

# drop duplicate claims (same replication, original and outcome).
# url_r and alt_identifier_r are included so multi-study rows sharing a doi_r aren't dropped.
dup_cols = ["doi_r", "alt_identifier_r", "url_r", "doi_o", "outcome"]
n_dups = batch.duplicated(dup_cols).sum()
batch = batch.drop_duplicates(dup_cols)
print(f"Dropped {n_dups} duplicate rows")

nanopubs_built = []   # (flora id, Nanopub) pairs
skipped = []          # (flora id, reason) pairs

for _, row in batch.iterrows():
    try:
        nanopubs_built.append((row["id"], build_nanopub(row, np_conf, multi_study_ids)))
    except ValueError as e:
        skipped.append((row["id"], str(e)))

n_multi = sum(i in multi_study_ids for i, _ in nanopubs_built)
print(f"Built {len(nanopubs_built)} nanopubs ({n_multi} multi-study), skipped {len(skipped)} rows")
skipped_df = pd.DataFrame(skipped, columns=["id", "reason"])
skipped_df.reason.value_counts()

print(nanopubs_built[0][1].provenance.serialize(format="turtle"))


Dropped 0 duplicate rows
Built 16 nanopubs (0 multi-study), skipped 0 rows
@prefix orcid: <https://orcid.org/> .
@prefix prov: <http://www.w3.org/ns/prov#> .

<http://purl.org/nanopub/temp/np/assertion> prov:wasAttributedTo <https://forrt.org/flora-replication-atlas/>,
        orcid:0000-0001-9911-526X .




In [109]:
# inspect a few before publishing
for flora_id, np_ in nanopubs_built:
    print(flora_id)
    print(np_.assertion.serialize(format="turtle"))

FLORA-000006
@prefix cito: <http://purl.org/spar/cito/> .
@prefix forrt: <https://w3id.org/sciencelive/o/terms/> .

<https://doi.org/10.1080/02671522.2019.1678062> a forrt:FORRT-Replication-Study ;
    cito:confirms <https://doi.org/10.1002/tea.21227> ;
    forrt:FORRT-Replication-Outcome "It replicates and extends previous findings, showing that among older students, levels of science capital remain patterned by gender, ethnicity, cultural capital and science set." .


FLORA-000007
@prefix cito: <http://purl.org/spar/cito/> .
@prefix forrt: <https://w3id.org/sciencelive/o/terms/> .

<https://doi.org/10.1371/journal.pone.0294325> a forrt:FORRT-Replication-Study ;
    cito:confirms <https://doi.org/10.1037/h0093718> ;
    forrt:FORRT-Replication-Outcome "We find an error rate of 33% for the standard length-of-line experiment which replicates the original findings by Asch (1951, 1955, 1956)." .


FLORA-000008
@prefix cito: <http://purl.org/spar/cito/> .
@prefix forrt: <https://w3id.org/s

# publish
- every published nanopub is logged to a CSV (separate files for test and real server)
- on a rerun, flora ids already in the log are skipped, so nothing is published twice
- short pause between publishes so the server isn't flooded

In [114]:
import csv, os, time
from datetime import datetime, timezone
from nanopub.definitions import TEST_NANOPUB_REGISTRY_URL

# the library swaps in the test server internally, so work out the real target here
SERVER = TEST_NANOPUB_REGISTRY_URL if np_conf.use_test_server else np_conf.use_server
LOG_FILE = "published_test.csv" if np_conf.use_test_server else "published.csv"
LOG_COLS = ["id", "nanopub", "server", "published_at"]

def already_published():
    if not os.path.exists(LOG_FILE):
        return set()
    return set(pd.read_csv(LOG_FILE)["id"])

def publish_batch(built, pause=0.5):
    done = already_published()
    todo = [(i, n) for i, n in built if i not in done]
    print(f"{len(built) - len(todo)} already published, publishing {len(todo)} to {SERVER}")

    new_file = not os.path.exists(LOG_FILE)
    with open(LOG_FILE, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=LOG_COLS)
        if new_file:
            writer.writeheader()
        for flora_id, np_ in todo:
            try:
                np_.publish()
            except Exception as e:
                print(f"{flora_id}: failed - {e}")
                continue
            writer.writerow({
                "id": flora_id,
                "nanopub": np_.source_uri,
                "server": SERVER,
                "published_at": datetime.now(timezone.utc).isoformat(),
            })
            f.flush()   # write each row immediately, so a crash doesn't lose the log
            print(f"{flora_id}: {np_.source_uri}")
            time.sleep(pause)

# uncomment when ready:
publish_batch(nanopubs_built)

6 already published, publishing 10 to https://registry.petapico.org/np/
FLORA-000015: https://w3id.org/np/RAAuj78J69ojSUBbDBLNiIANiWtbiJg99OouRyGRGYpnw
FLORA-000017: https://w3id.org/np/RAVK8D6_vfSlbwf2_-9vsd6WsDKM4mLmlRor_ox9oAK-g
FLORA-000018: https://w3id.org/np/RALfhhxzn7Gmo01uYrAkYuDiyfJTEMexUhooJgGcjGXZ4
FLORA-000019: https://w3id.org/np/RA3ZCKDoA2DDzs8mdJGltiPrBDvFGgUTArtPnhefCNPIU
FLORA-000020: https://w3id.org/np/RAj04rGZll4VKWPkY89iNHxYkHZHWdCV1RR40KOshMbXg
FLORA-000022: https://w3id.org/np/RADJt8QAJvtJbtLQQ6X_IchyCsfkaCCGxrlwWvH4BGhkI
FLORA-000023: https://w3id.org/np/RAAAIZHzwISFTrTMxaQflL3_qhQA3ON3Wbs5B1pk7t-4w
FLORA-000027: https://w3id.org/np/RAHJoVoiFRUN9nnVGGolOIpM_JRnGa7j5ZfE3nQsXoE2o
FLORA-000030: https://w3id.org/np/RA8FKYtREfo3PhHyDI-9k6dQxembNOCA3LWF1NvOgIZD8
FLORA-000036: https://w3id.org/np/RAGuugm-QiEGz5Ln1rVWPksMY6a6d76ebnwo_j_CSPluo


# How to retract if necessary
1. Build retractions

In [97]:
# import csv, os, time
# from datetime import datetime, timezone
# from nanopub import NanopubConf, load_profile
# from nanopub.templates.nanopub_retract import NanopubRetract

# # real network, NOT the test server
# real_conf = NanopubConf(
#     profile=load_profile(),
#     use_test_server=False,
# )

# RETRACT_LOG = "retracted.csv"

# published = pd.read_csv("published.csv")
# done = set(pd.read_csv(RETRACT_LOG)["nanopub"]) if os.path.exists(RETRACT_LOG) else set()
# to_retract = [u for u in published["nanopub"] if u not in done]
# print(f"{len(done)} already retracted, {len(to_retract)} to retract on {real_conf.use_server}")

# # build and inspect first
# retractions = [(uri, NanopubRetract(conf=real_conf, uri=uri)) for uri in to_retract]
# print(retractions[0][1].assertion.serialize(format="turtle"))


0 already retracted, 4 to retract on https://registry.petapico.org/np/
@prefix npx: <http://purl.org/nanopub/x/> .
@prefix orcid: <https://orcid.org/> .

orcid:0000-0001-9911-526X npx:retracts <https://w3id.org/np/RA373Yb0rpuiK3md7pwK0Z25nh5LF0IfMe4D2VpWKKCT0> .




2. Publish retractions

In [100]:
# def publish_retractions(retractions, pause=0.5):
#     new_file = not os.path.exists(RETRACT_LOG)
#     with open(RETRACT_LOG, "a", newline="") as f:
#         writer = csv.DictWriter(f, fieldnames=["nanopub", "retraction", "retracted_at"])
#         if new_file:
#             writer.writeheader()
#         for uri, r in retractions:
#             try:
#                 r.publish()
#             except Exception as e:
#                 print(f"{uri}: failed - {e}")
#                 continue
#             writer.writerow({
#                 "nanopub": uri,
#                 "retraction": r.source_uri,
#                 "retracted_at": datetime.now(timezone.utc).isoformat(),
#             })
#             f.flush()
#             print(f"retracted {uri} -> {r.source_uri}")
#             time.sleep(pause)

# publish_retractions(retractions)




retracted https://w3id.org/np/RA373Yb0rpuiK3md7pwK0Z25nh5LF0IfMe4D2VpWKKCT0 -> https://w3id.org/np/RAXwGoe3nrkCb03-8LjkQm5jmBHJFFmWEDivGheGXn-_s
retracted https://w3id.org/np/RAEVYCavRlzh_KhH8KpOM3Dpgn9zzU0yAkiQrOqTg-o6Q -> https://w3id.org/np/RAfsFNd_4eL1wnZkZh94a9l8ntHIeFUQVK13xfpTUT9VU
retracted https://w3id.org/np/RARlPrBSN8lyyuDlHMikGPVhHzk58tj-x7PXfOWHoDy34 -> https://w3id.org/np/RAvJb96gCXUVe6rO3PmEst219OFjcIEL9IhOvqoxI1dN4


TimeoutError: [Errno 60] Operation timed out

# Check all published nanopubs are in the log

In [ ]:
# from nanopub import NanopubClient

# CREATED_FROM_TEMPLATE = "https://w3id.org/np/o/ntemplate/wasCreatedFromTemplate"
# my_templates = [
#   "https://w3id.org/np/RA0-q9IdP_d6PPmIUeKjYLbm1wxsn_ga0gWTN1m-azxsI"
# ]

# client = NanopubClient()   # real network
# on_network = pd.DataFrame([
#     {"nanopub": r["np"], "date": r["date"], "template": t}
#     for t in my_templates
#     for r in client.find_nanopubs_with_pattern(pred=CREATED_FROM_TEMPLATE, obj=t)
# ])

# logged = set(pd.read_csv(LOG_FILE)["nanopub"])
# missing = on_network[~on_network["nanopub"].isin(logged)]
# missing


,nanopub,date,template
6,https://w3id.org/np/RAAuj78J69ojSUBbDBLNiIANiW...,2026-11-13T09:35:56.696Z,https://w3id.org/np/RA0-q9IdP_d6PPmIUeKjYLbm1w...


In [112]:
# from nanopub import Nanopub

# for uri in missing["nanopub"]:
#     np_ = Nanopub(source_uri=uri)
#     work = next(np_.assertion.subjects(RDF.type, FORRT["FORRT-Replication-Study"]))
#     match = flora_filtered[
#         flora_filtered["doi_r"].map(lambda d: pd.notna(d) and str(doi_uri(d)) == str(work))
#         | flora_filtered["url_r"].map(lambda u: pd.notna(u) and str(doi_uri(u)) == str(work))
#     ]
#     print(uri, "->", work, "->", match["id"].tolist())


https://doi.org/10.1002/1099-0879(200007)7:3<220::aid-cpp243>3.0.co;2-f does not look like a valid URI, trying to serialize this will break.
https://doi.org/10.1002/1520-6807(198607)23:3<271::aid-pits2310230308>3.0.co;2-y does not look like a valid URI, trying to serialize this will break.
https://doi.org/10.1002/1520-6629(199110)19:4<373::aid-jcop2290190409>3.0.co;2-j does not look like a valid URI, trying to serialize this will break.
https://doi.org/10.1002/(sici)1521-186x(1996)17:4<279::aid-bem4>3.0.co;2-0 does not look like a valid URI, trying to serialize this will break.
https://osf.io/v6zk8 or http://rpubs.com/catherinecthomas/final_replication does not look like a valid URI, trying to serialize this will break.


https://w3id.org/np/RAAuj78J69ojSUBbDBLNiIANiWtbiJg99OouRyGRGYpnw -> https://doi.org/10.1007/s11469-021-00695-9 -> ['FLORA-000015']


In [ ]:
# pd.DataFrame([{"id": "FLORA-…", "nanopub": "https://w3id.org/np/RA…",
#                "server": SERVER, "published_at": "…"}]
# ).to_csv(LOG_FILE, mode="a", header=False, index=False)
